In [4]:
import os
from trulens.core import Feedback, TruSession, Select
from trulens.providers.openai import OpenAI as ProviderOpenAI
from trulens.feedback import GroundTruthAgreement
from trulens.apps.basic import TruBasicApp
import numpy as np
import json
import re

Package trulens-apps-langgraph not present in requirements.


In [5]:
Target_location = "data/comparision/microsoft_phi-4_results.json"
model = '-'.join(Target_location.split("/")[-1].split('.')[0].split("_"))
print(f"Model: {model}")

Model: microsoft-phi-4-results


In [6]:
def clean_text(text):
    return re.sub(r"^```json\s*|\s*```$", "", text.strip())

In [7]:
with open("data/comparision/100_extracted_final.json", "r", encoding='utf-8') as file:
    ext_100 = json.load(file)

with open("data/comparision/100_raw.json", "r") as file:
    raw_100 = json.load(file)

with open(Target_location, "r") as file:
    target = json.load(file)

assert len(ext_100) == len(raw_100), "The two datasets must have the same length."
assert [case['cnr'] for case in ext_100] == list(raw_100.keys()), "CNRs must match between the two datasets."

raw_100 = [case for _, case in raw_100.items()]   
(ext_100[0], raw_100[0])  # Display the first elements of both datasets for verification

({'cnr': 'JHHC010253682018',
  'case details': 'Applicant applied for Bail-Cancellation. Is it a withdrawal application? No. Age of the accused is not provided. Health issues for the accused are None. There are no past criminal records of the accused. Statutes mentioned in the judgement are [Section 498A IPC]. Precedents mentioned in the judgement are None. Details of the incident are The petitioner is accused in Complaint Case No. 16 of 2016, registered for the offence under Section 498A of the Indian Penal Code. Arguments supporting the bail application are None. Arguments opposing the bail application are It appears that the petitioner has not complied with the order dated 20.02.2019, which shows that the petitioner is not interested to pursue the case.',
  'date_of_arrest': 'not provided.',
  'date_of_judgement': '13-03-2019.',
  'outcome': 'The outcome of the case is Bail cancelled. The bail conditions are Provisional bail granted to the petitioner vide order dated 24.08.2018, whi

In [8]:
fallback_keys = ['case', 'outcome', 'reasoning', 'date_of_arrest', 'date_of_judgement']
fallback_dict = {key: '' for key in fallback_keys}
failure_count = 0

new_target = []
for item in target:
    try:
        parsed = json.loads(clean_text(item['extracted_data']))
        new_target.append(parsed)
    except Exception:
        failure_count += 1
        new_target.append(fallback_dict.copy())

target = new_target
print(f"Failed to parse JSON {failure_count} times.")


Failed to parse JSON 0 times.


In [9]:
target[0]

{'case': 'Applicant applied for Bail-Cancellation.\nIs it a withdrawal application? No.\nAge of the accused is not provided.\nHealth issues for the accused are None.\nThere are no past criminal records of the accused.\nStatutes against the accused in the case are [Section 498A of the Indian Penal Code].\nPrecedents mentioned in the judgement are None.\nDetails of the incident are Petitioner is accused in Complaint Case No. 16 of 2016, registered for the offence under Section 498A of the Indian Penal Code.\nArguments supporting the bail application are None.\nArguments opposing the bail application are It appears that the petitioner has not complied with the order dated 20.02.2019, which shows that the petitioner is not interested to pursue the case.',
 'outcome': 'The outcome of the case is Bail cancelled. The bail conditions are Further, trial court is directed to take all coercive steps against the petitioner for his arrest.',
 'reasoning': 'In view of the above, provisional bail gra

In [10]:
cases = [
    {
        'case': raw_100[i],
        'ref': ext_100[i]['case details']+" "+ext_100[i]['outcome']+" \nReason: "+ext_100[i]['reasoning']+" \ndate of arrest: "+ext_100[i]['date_of_arrest']+" \ndate of judgment: "+ext_100[i]['date_of_judgement'],
        'res': target[i]['case']+" "+target[i]['outcome']+" \nReason: "+target[i]['reasoning']+" \ndate of arrest: "+target[i]['date_of_arrest']+" \ndate of judgment: "+target[i]['date_of_judgement']
        # 'res': clean_text(target[i]['extracted_data'])
    }
    for i in range(len(ext_100))
]

In [11]:
cases[0]

{'case': "IN THE HIGH COURT OF JHARKHAND AT RANCHI\nB.A. No. 6869 of 2018 2019:JHHC:8178\nSudama Chaudhary ….. Petitioner\nVersus\n1. The State of Jharkhand\n2. Lalita Devi ….. Opp. Parties\n---------\nCORAM: HON'BLE MR. JUSTICE ANANT BIJAY SINGH\n---------\nFor the Petitioner : Mr. Pranabesh Kr. Paul, Advocate.\nFor the State : A.P.P.\nFor the O.P. No. 2 : Mr. Manoj Kumar-II, Advocate.\n---------\n06 /Dated: 13/03/2019\nHeard learned counsel for the parties.\nPetitioner is accused in Complaint Case No. 16 of\n2016, registered for the offence under Section 498A of the\nIndian Penal Code.\nIt appears that under order dated 24.08.2018,\npetitioner was admitted on provisional bail, period not\nmentioned and notices were issued to opposite party no. 2.\nFurther, under order dated 16.01.2019, the order dated\n24.08.2018 was modified to the extent that petitioner was\nadmitted on provisional bail till 05.03.2019 and thereafter\nextended till 03.04.2019 and both the parties were directed\nto 

In [12]:
with open(f"data/comparision/{model}-eval.json", "w") as file:
    json.dump(cases, file, indent=4)

In [117]:
import re

result_files = list(os.listdir("results/"))
model_eval = {}

def extract_score(s):
    match = re.search(r'\b([1-9]|10)\b', s)
    return int(match.group(1)) if match else 0

for file in result_files:
    with open(f"results/{file}", "r") as f:
        data = json.load(f)
        sum_ = 0    
        for item in data:
            scores = item['all_responses']
            scores = [extract_score(s) for s in scores]
            score = 0.5*scores[0] + 0.3*scores[1] + 0.2*scores[2]
            sum_ = sum_ + score
        model_eval[file] = sum_ / len(data) if data else 0
        

In [118]:
model_eval

{'gpt4o_phi-4.json': 7.505999999999997,
 'gpt4o_deepseek.json': 6.086000000000001,
 'gpt4o_gemma.json': 6.813000000000001,
 'gpt4o_llama.json': 6.895000000000001,
 'gpt4o_mistral.json': 6.947999999999999}

In [41]:
with open('API_key.txt', 'r') as file:
    API_Key = file.readline().strip()

os.environ['OPENAI_API_KEY'] = API_Key

In [51]:
provider = ProviderOpenAI(model_engine="gpt-3.5-turbo")

f_factual_accuracy = Feedback(provider.relevance, name="Factual Accuracy").on_prompt("ref").on_response()

f_completeness = Feedback(
    provider.relevance_with_cot_reasons,
    name="Completeness and Comprehensiveness"
).on_prompt("ref").on_response()

✅ In Factual Accuracy, input prompt will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Completeness and Comprehensiveness, input prompt will be set to __record__.main_output or `Select.RecordOutput` .


In [43]:
gt_for_case = [
    {
        "query": item["case"],
        "expected_response": item["res"]
    }
    for item in cases
]

gt_case = GroundTruthAgreement(gt_for_case, provider=provider)
f_case_agreement = Feedback(gt_case.agreement_measure, name="Agreement with Case Text").on_input_output()


✅ In Agreement with Case Text, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Agreement with Case Text, input response will be set to __record__.main_output or `Select.RecordOutput` .


In [45]:
feedbacks = [f_factual_accuracy, f_completeness, f_case_agreement]

In [52]:
def predict(case_text: str) -> str:
    # Return your model output based on `case_text`
    return next(r["res"] for r in cases if r["case"] == case_text)

app = TruBasicApp(predict)

instrumenting <class 'trulens.apps.basic.TruWrapperApp'> for base <class 'trulens.apps.basic.TruWrapperApp'>
	instrumenting _call


In [54]:
session = TruSession()
    
df = session.run_app(
    app=app,
    inputs=[r["case"] for r in cases],         # main input = case
    reference_outputs=[r["ref"] for r in cases], # fed into selectors
    feedbacks=feedbacks
)

AttributeError: 'TruSession' object has no attribute 'run_app'